# Trening modelu bazowego

Ten notebook trenuje **bazowy model DeepMReye** na połączonym zbiorze danych składającym się z:
- Zbiorów 1–5: dane fMRI z prowadzonymi ruchami gałek ocznych
- Zbioru 7: dane fMRI w stanie spoczynku

Wynikowy model służy jako **wytrenowany szkielet (backbone)** dla kolejnych eksperymentów.

**Podział danych:** 80% trening (233 osoby), 20% test (60 osób), stratyfikowany per zbiór danych.

In [ ]:
import os
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
import tensorflow as tf
tf.keras.backend.clear_session()
import pandas as pd
from deepmreye import analyse, architecture, preprocess, train
from deepmreye.util import data_generator, model_opts, util
import numpy as np
gpus = tf.config.experimental.list_physical_devices('GPU')
tf.config.experimental.set_memory_growth(gpus[0], True)

## Funkcje pomocnicze

`create_holdout_generators` opakowuje funkcję `data_generator.create_generators` z biblioteki DeepMReye.
Przyjmuje listę folderów ze zbiorami danych, lub jawne listy ścieżek `train_list` / `test_list`.
Zwraca generatory treningowe i testowe oraz generatory per-podmiot używane podczas ewaluacji.

In [ ]:
def create_holdout_generators(datasets=None, train_split=0.6, train_list=None, test_list=None, **args):
    if train_list is not None and test_list is not None:
        full_training_list = train_list
        full_testing_list = test_list
    else:
        full_training_list, full_testing_list = list(), list()
        for fn_data in datasets:
            this_file_list = [fn_data + p for p in os.listdir(fn_data)]
            np.random.shuffle(this_file_list)
            split_idx = int(train_split * len(this_file_list))
            this_training_list = this_file_list[0:split_idx]
            this_testing_list = this_file_list[split_idx:]
            full_training_list.extend(this_training_list)
            full_testing_list.extend(this_testing_list)

    (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
    ) = data_generator.create_generators(full_training_list, full_testing_list, **args)

    return (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
        full_testing_list,
        full_training_list,
    )

## Konfiguracja modelu

Domyślne hiperparametry DeepMReye są ładowane za pomocą `model_opts.get_opts()`.
Kluczowe wartości domyślne: `lr=2e-05`, `dropout_rate=0.1`, `loss_euclidean=1`, `loss_confidence=0.1`.

In [ ]:
opts = model_opts.get_opts()
print(opts)

In [ ]:
opts['epochs']=1
opts['steps_per_epoch']=5000
opts['validation_steps']=100
print(opts)

## Wczytywanie danych

Wszystkie przetworzone pliki podmiotów (`.npz`) są zbierane rekurencyjnie z katalogu z danymi.
Pliki są grupowane według zbioru danych (nazwa folderu nadrzędnego), aby podział 80/20 był stosowany **osobno dla każdego zbioru**,
zapobiegając nadreprezentacji lub niedoreprezentacji któregokolwiek ze zbiorów w treningu lub teście.

In [ ]:
all_files=[]
for root, dirs, files in os.walk('/mnt/compneuro/deepmreye_finetuning/processed_data/'):
    for name in files:
        all_files.append(os.path.join(root,name))
        print(os.path.join(root,name))

In [ ]:
datasets = {}
for f in all_files:
    dataset_name = os.path.basename(os.path.dirname(f))
    datasets.setdefault(dataset_name, []).append(f)

train_list_rs, test_list_rs = [], []
for dataset_name, files in datasets.items():
    files = sorted(files)                
    np.random.shuffle(files)             
    split_idx = int(0.8 * len(files))   
    train_list_rs.extend(files[:split_idx])
    test_list_rs.extend(files[split_idx:])

print("Total train:", len(train_list_rs))
print("Total test:", len(test_list_rs))

## Zapisywanie i wczytywanie podziału na zbiór treningowy i testowy

Podział jest zapisywany do plików `.txt` w celu zapewnienia **reprodukowalności** - te same osoby są używane we wszystkich kolejnych eksperymentach.

In [ ]:
#np.savetxt('/mnt/compneuro/deepmreye_finetuning/sleepybrain/train_list.txt',train_list_rs,fmt="%s")
#np.savetxt('/mnt/compneuro/deepmreye_finetuning/sleepybrain/test_list.txt',test_list_rs,fmt="%s")

In [ ]:
train_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/sleepybrain/train_list.txt',dtype=str)
test_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/sleepybrain/test_list.txt',dtype=str)

## Tworzenie generatorów danych

Ścieżki są konwertowane na ścieżki bezwzględne.
Podczas treningu stosowana jest augmentacja danych.
`mixed_batches=True` zapewnia, że każda paczka (batch) zawiera próbki z wielu podmiotów, co stabilizuje trening.

In [ ]:
train_list_rs_absolute = []
for f in train_list_rs:
    if f.startswith('./processed_data/'):
        absolute_path = f.replace('./processed_data/', '/mnt/compneuro/deepmreye_finetuning/processed_data/')
    else:
        absolute_path = f
    train_list_rs_absolute.append(absolute_path)

test_list_rs_absolute = []
for f in test_list_rs:
    if f.startswith('./processed_data/'):
        absolute_path = f.replace('./processed_data/', '/mnt/compneuro/deepmreye_finetuning/processed_data/')
    else:
        absolute_path = f
    test_list_rs_absolute.append(absolute_path)

generators_both = create_holdout_generators(
    train_list=train_list_rs_absolute, 
    test_list=test_list_rs_absolute, 
    batch_size=opts['batch_size'], 
    augment_list=(
        (opts['rotation_x'], opts['rotation_y'], opts['rotation_z']), 
        opts['shift'], 
        opts['zoom']
    ), 
    mixed_batches=True
)

## Trening modelu bazowego

Model DeepMReye jest trenowany od zera na połączonym zbiorze danych.
Wytrenowany model jest automatycznie zapisywany na dysku (`save=True`).

In [ ]:
(model_both, model_inference_both) = train.train_model(dataset='both_eyes_RS-in-the-model_short', generators=generators_both, opts=opts, workers=4,
                                            return_untrained=False, verbose=1, save=True)

## Wczytywanie wytrenowanego modelu do ewaluacji

Tworzona jest pusta powłoka modelu o tej samej architekturze, a następnie wczytywane są zapisane wagi.
Generator testowy jest konfigurowany z podmiotami z wydzielonego zbioru testowego (60 osób ze wszystkich zbiorów danych).

In [ ]:
generators_both = data_generator.create_generators(test_list_rs_absolute,
                                              test_list_rs_absolute)
generators_both = (*generators_both, test_list_rs_absolute, test_list_rs_absolute
              )  
(model_both, model_inference_both) = train.train_model(dataset="prediction_on_both_eyes",
                                             generators=generators_both,
                                             opts=opts,
                                             return_untrained=True)
model_inference_both.load_weights('/mnt/compneuro/deepmreye_finetuning/modelinference_both_eyes_RS-in-the-model.h5')

## Ewaluacja modelu

Model jest ewaluowany per podmiot na wydzielonym zbiorze testowym.
Raportowane metryki: **korelacja Pearsona r** (X i Y), **R²** oraz **błąd euklidesowy** (w znormalizowanych jednostkach spojrzenia).

In [ ]:
(evaluation_both, scores_both) = train.evaluate_model(dataset='both_eyes', model=model_inference_both, generators=generators_both, save=True, model_description='', verbose=2)

## Wizualizacja predykcji

Interaktywny wykres z suwakiem pokazujący przewidywane i rzeczywiste szeregi czasowe spojrzenia per podmiot.

In [ ]:
fig = analyse.visualise_predictions_slider(evaluation_both, scores_both, color="rgb(0, 150, 175)", bg_color="rgb(255,255,255)",
                                   ylim=[-2, 2],)
fig.show()